In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
import umap
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GridSearchCV

In [ ]:
file_path=r"...\implementation\notebook\output_filename2.csv"

In [ ]:
df_cleaned = pd.read_csv(file_path)

In [ ]:
df_cleaned.columns


### Encodage des colonnes

In [ ]:
colonnes_label = ['Gender', 'Age_Category', 'BMI_Category']
le_dict = {}

for col in colonnes_label:
    le = LabelEncoder()
    df_cleaned[col] = le.fit_transform(df_cleaned[col])
    le_dict[col] = le  


In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns=['Type of Diet'], drop_first=True)


In [ ]:
X = df_cleaned.drop(['Scoring', 'No'], axis=1)
y = df_cleaned['Scoring']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


#### clean

In [ ]:
cols_to_clean = [
    'Iron (mg)', 'Sodium (mg)', 'Calcium (mg)', 'Phosphorus (mg)',
    'Potassium (mg)', 'Magnesium (mg)', 'Zinc (mg)', 'Iodine (?g)',
    'Selenium (?g)', 'Copper (mg)', 'Manganese (mg)', 'omega 3',
    'Vitamin A (?g)', 'Vitamin D (?g)', 'Vitamin E (mg)',
    'wit. B1 - Tiamina', 'wit. B2 - Riboflavin', 'Vitamin B3 (mg)',
    'Vitamin B5 (mg)', 'Vitamin B6 (mg)', 'Vitamin B7 (?g)',
    'Vitamin B12 (?g)', 'Vitamin C (mg)', 'Vitamin K (?g)', 'Omega 6 (g)',
    'Cholesterol (mg/day)', 'Dietary Fiber (g)'
]

for col in cols_to_clean:
    # Supprimer les virgules
    X_train[col] = X_train[col].str.replace(',', '', regex=False)
    X_test[col] = X_test[col].str.replace(',', '', regex=False)
    
    # Convertir en float
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))  # toujours bas? sur X_train


### <center> RANDOM FOREST </center>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Matrice de confusion pour le mod?le Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
disp_rf.plot(cmap='Greens')
plt.title("?? Matrice de confusion - Random Forest")
plt.show()


### <center> XGBoost  </center>

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report



In [ ]:
# Cr?ation du mod?le
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)



In [ ]:
# Entra?nement
xgb_model.fit(X_train, y_train)

# Pr?diction
y_pred_xgb = xgb_model.predict(X_test)



In [ ]:
# ?valuation
print("?? R?sultats du mod?le XGBoost")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

# Pr?dictions
y_pred_xgb = xgb_model.predict(X_test)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("?? Matrice de confusion - XGBoost")
plt.show()


### Hyperparameters

In [ ]:
# D?finir le mod?le de base
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Grille d'hyperparam?tres
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}




In [ ]:
# Recherche des meilleurs param?tres
grid_search = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    cv=3, scoring='accuracy', 
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)


In [ ]:
# Meilleur mod?le
best_xgb = grid_search.best_estimator_
print("?? Meilleurs param?tres XGBoost : ", grid_search.best_params_)



In [ ]:
from sklearn.metrics import classification_report
y_pred_best = best_xgb.predict(X_test)
print("?? Rapport de classification apr?s tuning :")
print(classification_report(y_test, y_pred_best))


### Importance des variables

In [ ]:
importances = best_xgb.feature_importances_
features = X_train.columns

# Tri des variables par importance
sorted_idx = importances.argsort()
plt.figure(figsize=(10, 8))
plt.barh(features[sorted_idx], importances[sorted_idx])
plt.title("?? Importance des variables - XGBoost")
plt.xlabel("Score d'importance")
plt.tight_layout()
plt.show()


In [ ]:
import pickle

# Sauvegarder le meilleur mod?le dans un fichier
with open('best_xgb_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

print("? Mod?le sauvegard? dans 'best_xgb_model.pkl'")


In [ ]:
# Charger le mod?le XGBoost sauvegard?
path=r"C:\Users\PC\Desktop\ESISA\4 eme - Semestre 2\Data Mining\Projet sant? 2\Projet sant? 2\implementation\notebook\best_xgb_model.pkl"
with open(path, 'rb') as f:
    best_xgb = pickle.load(f)


In [ ]:
importances = best_xgb.feature_importances_
features = X_train.columns

# Combine features and importances into a DataFrame for easier manipulation
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})

# Sort the DataFrame by importance in descending order
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Get the top 12 features
top_12_features = feature_importance_df.head(12)

# Plot only the top 12 features
plt.figure(figsize=(10, 8))
plt.barh(top_12_features['Feature'][::-1], top_12_features['Importance'][::-1])  # [::-1] to reverse for proper ordering
plt.title("?? Top 12 Important Features - XGBoost")
plt.xlabel("Score d'importance")
plt.tight_layout()
plt.show()

In [ ]:
print(top_12_features)

## TOP 12 

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import pandas as pd
import pickle

# Liste des 12 meilleures variables
top_features = [
    "Cholesterol LDL_x",
    "Fat (%)",
    "Carbohydrates (%)",
    "wit. B2 - Riboflavin",
    "Vitamin B5 (mg)",
    "Copper (mg)",
    "Zinc (mg)",
    "Vitamin D (?g)",
    "BMI",
    "Protein_Fat_Ratio",
    "Selenium (?g)",
    "Manganese (mg)"
]

# S?lection uniquement des top features
X_train_reduced = X_train[top_features]
X_test_reduced = X_test[top_features]

#  D?finir le mod?le de base
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

#  Grille d'hyperparam?tres
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

#  Recherche des meilleurs param?tres
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_reduced, y_train)

#  Meilleur mod?le entra?n?
best_xgb = grid_search.best_estimator_
print(" Meilleurs param?tres XGBoost : ", grid_search.best_params_)

# ?valuation
y_pred_best = best_xgb.predict(X_test_reduced)
print(" Rapport de classification apr?s tuning :")
print(classification_report(y_test, y_pred_best))

#  Importance des variables (top 12)
importances = best_xgb.feature_importances_
features = X_train_reduced.columns

#  Affichage des importances
sorted_idx = importances.argsort()
plt.figure(figsize=(10, 8))
plt.barh(features[sorted_idx], importances[sorted_idx])
plt.title(" Importance des variables - XGBoost (top 12)")
plt.xlabel("Score d'importance")
plt.tight_layout()
plt.show()

#  Sauvegarde du mod?le
with open("best_xgb_model_top12.pkl", "wb") as f:
    pickle.dump(best_xgb, f)

print(" Nouveau mod?le sauvegard? dans 'best_xgb_model_top12.pkl'")


In [ ]:
import pandas as pd
import pickle

# ?? Charger le mod?le entra?n?
with open("best_xgb_model_top12.pkl", "rb") as f:
    best_xgb = pickle.load(f)

# ?? Exemple de donn?e test (valeurs fictives ? remplace-les par les tiennes)
donnee_test = pd.DataFrame([{
    "Cholesterol LDL_x": 2.3,
    "Fat (%)": 30,
    "Carbohydrates (%)": 50,
    "wit. B2 - Riboflavin": 1.2,
    "Vitamin B5 (mg)": 4.5,
    "Copper (mg)": 0.9,
    "Zinc (mg)": 10.2,
    "Vitamin D (?g)": 5.3,
    "BMI": 24.8,
    "Protein_Fat_Ratio": 0.9,
    "Selenium (?g)": 60.0,
    "Manganese (mg)": 2.1
}])

# ?? Pr?diction
prediction = best_xgb.predict(donnee_test)[0]

# ? Interpr?tation
if prediction == 0:
    print("? Pr?diction : Faible risque")
else:
    print("?? Pr?diction : Haut risque")


In [ ]:
import pandas as pd
import pickle
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ?? Charger le mod?le sauvegard?
with open("best_xgb_model_top12.pkl", "rb") as f:
    best_xgb = pickle.load(f)

# ?? Liste des 12 variables utilis?es
top_features = [
    "Cholesterol LDL_x",
    "Fat (%)",
    "Carbohydrates (%)",
    "wit. B2 - Riboflavin",
    "Vitamin B5 (mg)",
    "Copper (mg)",
    "Zinc (mg)",
    "Vitamin D (?g)",
    "BMI",
    "Protein_Fat_Ratio",
    "Selenium (?g)",
    "Manganese (mg)"
]

X_test_reduced = X_test[top_features]
y_test_reduced = y_test

y_pred = best_xgb.predict(X_test_reduced)

print("?? Comparaison entre vraies classes et pr?dictions :\n")
for i in range(len(y_test_reduced)):
    vrai = y_test_reduced.iloc[i]
    predit = y_pred[i]
    resultat = "?? Correct" if vrai == predit else "? Incorrect"
    interpretation = "Faible" if predit == 0 else "Haut Risque"
    print(f"Exemple {i+1:02d} | Vrai: {vrai} | Pr?dit: {predit} ({interpretation}) ? {resultat}")
